# basic anno turtorial

This is a demonstration of the downstream analysis after using `vampire anno` to annotate a TR locus across population.

> [!NOTE]
> You can download this notebook by clicking the **Download** button in the upper-right corner of this page and selecting the `.ipynb` format.

In [1]:
import numpy as np
import vampire as vp
import logging
import plotly.io as pio
pio.renderers.default = "notebook_connected"

Set up logging for intermediate information.

In [2]:
vampire_logger = logging.getLogger("vampire")
vampire_logger.setLevel(logging.INFO) # set level to "DEBUG", "INFO", "WARNING"
if not vampire_logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(
        logging.Formatter(
            "[%(asctime)s] %(message)s",
            datefmt="%Y-%m-%d %H:%M:%S",
        )
    )
    vampire_logger.addHandler(handler)

Set the default plot style for all the plots in this notebook. You can set `font_size`, `font_family`, `line_width`, `height`, `width`, `showgrid` to set default parameters.

In [3]:
vp.anno.pl.set_default_plotstyle()

The dataset `wdr7_hprc` used in this tutorial was collected from the [Yang et al., 2026](https://www.biorxiv.org/content/10.1101/2025.06.15.659631v1). It contains annotations of a 69-bp motif VNTR locus (`chr18:57,226,379–57,227,527` in T2T-CHM13v2.0) located within an intron of the *`WDR7`* gene across 95 human haplotype assemblies, including 47 individuals from Phase 1 of the `Human Pangenome Reference Consortium (HPRC)` and the `T2T-CHM13v2.0` reference genome.

In [4]:
adata = vp.datasets.wdr7_hprc()
adata

AnnData object with n_obs × n_vars = 95 × 24
    obs: 'length', 'copy_number', 'score'
    var: 'motif', 'motif_length', 'copy_number', 'label'
    uns: 'motif_array', 'orientation_array', 'sequence'
    varp: 'motif_distance', 'rc_motif_distance'

## Data Structure

### `X`
Motif abundance / copy-number matrix with shape `(n_obs × n_var)`.

```python
X[i, j] = copy number of motif j in sample i
```

---

### `obs`
Sample-level metadata with shape `(n_obs × metadata)`.

| Column | Type | Description |
|---|---|---|
| `length` | `int` | Length of the TR sequence |
| `copy_number` | `float` | Total copy number of the locus |
| `score` | `int` | Annotation score |

---

### `var`
Motif-level metadata with shape `(n_var × metadata)`.

| Column | Type | Description |
|---|---|---|
| `motif` | `str` | Motif sequence |
| `motif_length` | `int` | Length of the motif |
| `copy_number` | `float` | Total motif copy number across samples |
| `label` | `str` | Motif label |

---

### `varp`
Motif-level pairwise relations with shape `(n_var × n_var)`.

| Key | Type | Description |
|---|---|---|
| `motif_distance` | `int` | Pairwise motif distance matrix |
| `rc_motif_distance` | `int` | Reverse-complement motif distance matrix |

---

### `uns`
Unstructured genomic annotations not aligned to `X`.

| Key | Type | Description |
|---|---|---|
| `sequence` | `Dict[str, str]` | Original TR sequences |
| `motif_array` | `Dict[str, List[str]]` | Motif arrays |
| `orientation_array` | `Dict[str, List[str]]` | Motif orientation arrays |

## Adding ancestry information

In [5]:
adata.obs["sample"] = (
    adata.obs.index
    .astype(str)
    .str.replace(r"[\.][12]$", "", regex=True)
)

ancestry = vp.datasets.ancestry()
adata.obs["ancestry"] = adata.obs["sample"].map(ancestry)

In [6]:
adata.obs

,length,copy_number,score,sample,ancestry
sample,,,,,
CHM13Y,1147,16.7,2276,CHM13Y,NA
HG002.1,666,9.7,1314,HG002,EUR
HG002.2,2183,31.7,4357,HG002,EUR
HG00438.1,1905,27.7,3783,HG00438,EAS
HG00438.2,3080,44.7,6151,HG00438,EAS
...,...,...,...,...,...
NA19240.2,732,10.7,1446,NA19240,AFR
NA20129.1,3977,57.7,7909,NA20129,AFR
NA20129.2,389,5.7,769,NA20129,AFR


## Visualization across population

In [7]:
# sort samples by copy number
sample_order = list(
    adata.obs.sort_values(by="copy_number").index
)

# visualize the TR locus across population
# motifs exceeding the color limit will be displayed in black
vp.anno.pl.waterfall(
    adata,
    colormap = "rainbow", # choose from "rainbow", "glasbey" and "sequential", or you can specify colormap yourself
    figsize = (600, 800),
    sample_order = sample_order, # apply custom sample ordering
    track_name_dx = -0.01, # adjust horizontal position of sample labels
    
    # additional keyword arguments passed to plotly.graph_objects.Figure.update_layout(), including margin, font, etc.
    margin=dict(l=120), # leave more margin on the left for sample labels
    font=dict(size=8) # adjust to avoid label overlapping
)

[2026-05-17 13:08:07] Number of id is larger then number of colors in default colormap, using black to represent remaining motifs


Generate a legend for the waterfall plot. Use `waterfall_legend()` with the same parameters as `waterfall()` except the `color` parameter. To display motif sequences, set `color` as `"motif"`. To display motif IDs, set `color` as `"id"`.

In [8]:
vp.anno.pl.waterfall_legend(
    adata,
    color="motif", 
    colormap = "rainbow",
    figsize = (1200, 600),
    track_name_dx = -0.01,
    margin = dict(l=120),
)

## Motif Variation Pattern

By default, in `logo()`, conserved positions are shown in gray. Set to None to disable this feature or specify a color to highlight conserved positions

In [9]:
vp.anno.pl.logo(
    adata,
    feature = "information", # choose from "count", "probability" and "information"
    figsize = (900, 200),
)

In [10]:
vp.anno.pl.logo(
    adata,
    feature = "information", # choose from "count", "probability" and "information"
    conserved_color = None, # 
    figsize = (900, 200),
)

## Motif Abundance 

Tandem repeats often exhibit substantial sequence variation at the motif level across populations. Beyond differences in total copy number, individual alleles may vary in motif composition/abundance, and organization, reflecting diverse mutational and evolutionary processes such as motif duplication, substitution, and local rearrangement. Population-level motif abundance analysis aims to characterize these patterns by quantifying the relative abundance of distinct motifs across samples and identifying shared or divergent repeat architectures among individuals.

In [11]:
vp.anno.pl.motif_abundance_heatmap(
    adata,
    cluster_rows = True,
    cluster_cols = True,
    standard_scale = "obs", # use Max-Min normalization on "obs" row or "var" column， or set to None to disable normalization
    row_annotation = adata.obs["ancestry"].to_list(), # add row annotation
    figsize = (500, 1200),
    font = dict(size=8) # adjust to avoid label overlapping
)

Except for heatmap visualization, motif composition patterns can also be quantitatively summarized using `principal component analysis` (`PCA`). By representing each tandem repeat allele as a motif composition vector, PCA projects population-level variation into a low-dimensional space, allowing major axes of motif composition diversity to be identified and visualized. Samples with similar motif usage patterns tend to cluster together in the embedding space, while distinct haplotypes or repeat architectures may separate along principal components. This provides a compact and interpretable representation of motif composition variation across populations.

In [12]:
vp.anno.tl.motif_abundance_pca(
    adata,
    n_components = 10,
)

[2026-05-17 13:08:09] PCA on motif abundance: 10 components. Explained variance: 36.0%, 18.0%, 13.9%, 8.6%, 6.1%, 4.9%, 2.8%, 2.5%, 1.9%, 1.2%


AnnData object with n_obs × n_vars = 95 × 24
    obs: 'length', 'copy_number', 'score', 'sample', 'ancestry'
    var: 'motif', 'motif_length', 'copy_number', 'label'
    uns: 'motif_array', 'orientation_array', 'sequence', 'motif_abundance_pca'
    obsm: 'X_motif_abundance_pca'
    varm: 'motif_abundance_PCs'
    varp: 'motif_distance', 'rc_motif_distance'

In [13]:
vp.anno.pl.motif_abundance_pca_variance(
    adata, 
    figsize = (700, 600),
)

In [14]:
vp.anno.pl.motif_abundance_pca(
    adata,
    shape_by = "ancestry",
    color_by = "copy_number",
    components = (1, 2), # choose which principal components to plot
    )

In [15]:
# loading TODO

## Haplotype calling

`vp.anno.tl.align()` performs a `multiple sequence alignment` (`MSA`) of motif arrays across samples using a greedy progressive alignment strategy. It first builds a substitution matrix from pairwise motif distances and lengths, where similar motifs receive higher alignment scores. Starting from individual sequences as single-sequence profiles, it iteratively identifies the closest pair, aligns their consensus sequences via the `Needleman–Wunsch` algorithm with affine gap penalties, and merges the corresponding profiles by inserting gap columns. This process repeats until all samples are merged into a single MSA. 

If refinement is enabled (default), each original sequence is then realigned against the final consensus to correct accumulated early errors, yielding cleaner results. The aligned motif arrays, orientation arrays, and consensus sequence are stored in `.uns["aligned_motif_array"]`, `.uns["aligned_orientation_array"]` and `.uns["aligned_consensus"]`.

In [16]:
vp.anno.tl.align(
    adata,
    match_score = 2,
    mismatch_penalty = -3,
    gap_open_penalty = -5,
    gap_extend_penalty = -1,
)

[2026-05-17 13:08:17] Aligned 95 samples. Original lengths: [17, 10, 32, 28, 45, 27, 27, 10, 26, 24, 29, 18, 10, 7, 2, 18, 17, 31, 33, 2, 19, 30, 10, 7, 34, 24, 16, 13, 5, 16, 44, 24, 20, 2, 43, 4, 10, 18, 29, 27, 27, 39, 38, 8, 44, 25, 9, 26, 51, 54, 7, 26, 35, 5, 34, 6, 3, 20, 6, 4, 4, 13, 55, 54, 19, 6, 7, 9, 17, 45, 31, 11, 11, 7, 16, 7, 5, 16, 3, 8, 5, 12, 17, 9, 15, 5, 3, 4, 3, 9, 11, 58, 6, 17, 8]. Aligned length: 58.


AnnData object with n_obs × n_vars = 95 × 24
    obs: 'length', 'copy_number', 'score', 'sample', 'ancestry'
    var: 'motif', 'motif_length', 'copy_number', 'label'
    uns: 'motif_array', 'orientation_array', 'sequence', 'motif_abundance_pca', 'aligned_motif_array', 'aligned_orientation_array', 'aligned_consensus'
    obsm: 'X_motif_abundance_pca'
    varm: 'motif_abundance_PCs'
    varp: 'motif_distance', 'rc_motif_distance'

Show aligned motif arrays by setting feature = `"aligned_motif"` in `waterfall()` plot.

In [17]:
vp.anno.pl.waterfall(
    adata,
    feature = "aligned_motif",
    colormap = "rainbow",
    sample_order = sample_order,
    figsize = (600, 800),
    margin = dict(l=120),
    font = dict(size=8),
    track_name_dx = -0.01
)

`vp.anno.tl.haplotype_neighbor()` builds a fused `k-nearest-neighbour` (`KNN`) graph from aligned motif arrays for downstream haplotype clustering. It computes one or more pairwise distance matrices:

- structural distance (average motif distance over aligned non-gap positions)
- composition distance (Jensen–Shannon divergence between row-normalised motif frequency vectors)
- CNV distance (gap count weighted by a gap penalty)


And then it converts each into an adaptive kNN similarity graph using a Gaussian kernel with a per-node local scale. The individual graphs are symmetrised and then fused by element-wise aggregation (max (default), mean, or min); the resulting connectivities are stored in `.obsp["haplotype_connectivities"]`.

`vp.anno.tl.haplotype_leiden()` clusters samples into haplotypes by running the Leiden community-detection algorithm on the fused graph produced by `vp.anno.tl.haplotype_neighbor()`. After obtaining cluster labels, it constructs a consensus motif sequence for each haplotype by taking the most frequent non-gap motif at every alignment position among cluster members. The haplotype labels (e.g., H1, H2) are stored in `.obs["haplotype"]`, and the consensus sequences are saved in `.uns["haplotype_consensus"]`.


In [18]:
vp.anno.tl.haplotype_neighbor(
    adata,
    metrics = ["structural", "composition", "cnv"],
)
vp.anno.tl.haplotype_leiden(
    adata,
    resolution = 1
)                                                             

[2026-05-17 13:08:18] Computing distance matrices for metrics: ['structural', 'composition', 'cnv']
[2026-05-17 13:08:18] Computed composition distance from adata.X
[2026-05-17 13:08:18] Stored structural distance matrix in obsp['haplotype_structural_distance']
[2026-05-17 13:08:18] Stored cnv distance matrix in obsp['haplotype_cnv_distance']
[2026-05-17 13:08:18] Stored composition distance matrix in obsp['haplotype_composition_distance']
[2026-05-17 13:08:18] Building adaptive kNN graphs (k=9, n=95)
[2026-05-17 13:08:18] Built kNN graph for 'structural': 703 undirected edges
[2026-05-17 13:08:18] Built kNN graph for 'composition': 606 undirected edges
[2026-05-17 13:08:18] Built kNN graph for 'cnv': 591 undirected edges
[2026-05-17 13:08:18] Fusing 3 graphs using 'max'
[2026-05-17 13:08:18] Fused graph: 1456 undirected edges
[2026-05-17 13:08:18] Haplotype neighbour graph ready: 95 nodes, 1456 edges, 1 component(s)
[2026-05-17 13:08:18] Running Leiden clustering (resolution=1.00, ran

AnnData object with n_obs × n_vars = 95 × 24
    obs: 'length', 'copy_number', 'score', 'sample', 'ancestry', 'haplotype'
    var: 'motif', 'motif_length', 'copy_number', 'label'
    uns: 'motif_array', 'orientation_array', 'sequence', 'motif_abundance_pca', 'aligned_motif_array', 'aligned_orientation_array', 'aligned_consensus', 'haplotype', 'haplotype_consensus'
    obsm: 'X_motif_abundance_pca'
    varm: 'motif_abundance_PCs'
    obsp: 'haplotype_structural_distance', 'haplotype_cnv_distance', 'haplotype_composition_distance', 'haplotype_connectivities'
    varp: 'motif_distance', 'rc_motif_distance'

`vp.anno.pl.haplotype_distance_heatmap()` visualises a pairwise distance matrix produced by `vp.anno.tl.haplotype_neighbor()` as an interactive heatmap. It retrieves the requested (`structural`, `cnv`, or `composition`) distance matrix from `.obsp`, optionally reorders rows and columns by haplotype label so that members of the same haplotype appear adjacent.

In [19]:
vp.anno.pl.haplotype_distance_heatmap(
    adata,
    metric = "structural", # choose from "structural", "composition" or "cnv"
    cluster = True,
    figsize = (1000, 1100),
    vmax = 1.5,
    font = dict(size=8)
)

Then let's group the samples by haplotype and mark the haplotype groups on waterfall plot.

In [ ]:
adata_plot = adata.copy()

# sort haplotype
adata_plot.obs["_hap_num"] = (
    adata_plot.obs["haplotype"]
    .astype(str)
    .str.extract(r"(\d+)")
    .astype(int)
)
# sort samples by haplotype and copy number
sample_order = (
    adata_plot.obs
    .sort_values(["_hap_num", "copy_number"])
    .index
    .tolist()
)

# plot
fig = vp.anno.pl.waterfall(
    adata_plot,
    feature = "motif", # "aligned_motif" or "motif"
    colormap = "rainbow",
    sample_order = sample_order,
    figsize = (700, 900),
    margin = dict(l=120, r=100),
    font = dict(size=8),
    track_name_dx = -0.01,
)

# build ordered metadata table
obs_sorted = adata_plot.obs.loc[sample_order].copy()
obs_sorted["_pos"] = np.arange(len(obs_sorted))

hap_blocks = (
    obs_sorted
    .groupby("haplotype", sort=False, observed=True)["_pos"]
    .agg(["min", "max"])
)

n = len(sample_order)

for hap, row in hap_blocks.iterrows():
    start = row["min"] + 0.5
    end = row["max"] + 0.5
    mid = (start + end) / 2

    y0 = 1 - (end + 0.5) / n
    y1 = 1 - (start - 0.5) / n
    y_mid = 1 - (mid + 0.5) / n

    # main vertical line
    fig.add_shape(
        type="line",
        xref="paper",
        yref="paper",
        x0=1.01,
        x1=1.01,
        y0=y0,
        y1=y1,
        line=dict(width=1.5),
    )

    # top short horizontal line
    fig.add_shape(
        type="line",
        xref="paper",
        yref="paper",
        x0=1.01,
        x1=1.02,
        y0=y1,
        y1=y1,
        line=dict(width=1.5),
    )

    # bottom short horizontal line
    fig.add_shape(
        type="line",
        xref="paper",
        yref="paper",
        x0=1.01,
        x1=1.02,
        y0=y0,
        y1=y0,
        line=dict(width=1.5),
    )

    fig.add_annotation(
        xref="paper",
        yref="paper",
        x=1.03,
        y=y_mid,
        text=f"{hap}",
        showarrow=False,
        xanchor="left",
        yanchor="middle",
        font=dict(size=14),
    )

fig.show()

In [21]:
vp.anno.pl.copy_number_violin(
    adata,
    group_by = "haplotype",
    motif = 2,
    show_box = True,
    show_points = True,
    figsize = (500, 500),
)

In [22]:
vp.anno.pl.copy_number_violin(
    adata,
    group_by = "ancestry",
    motif = 2,
    show_box = True,
    show_points = True,
    figsize = (500, 500),
)

In [23]:
vp.anno.pl.copy_number_stacked_violin(
    adata,
    group_by = "haplotype",
    motifs = None,
    colorscale = "Reds", # Reds, Blues
    figsize = (500, 900)
)